<a href="https://colab.research.google.com/github/mesata/Store-Sales-Forecasting/blob/main/Arima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/ML final project'

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"it created: {PROJECT_PATH}")

%cd {PROJECT_PATH}

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/13YZMRWkS2eT6kmRi5e5vzlG7Yl3Vbt3z/ML final project


In [2]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import mlflow
import mlflow.sklearn

In [4]:
!pip install dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.7 MB/s eta 0:00:00


In [5]:
import dagshub
dagshub.init(repo_owner='mesata', repo_name='Walmart---Store-Sales-Forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8f292266-9807-492c-829c-3682d4b7a236&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=402dceea2d451d5494053e2f12b03b047b017d9799e7f956b311afe9ab86eb24




Accessing as mesata

Initialized MLflow to track repo "mesata/Walmart---Store-Sales-Forecasting"

Repository mesata/Walmart---Store-Sales-Forecasting initialized!

In [6]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip')
features_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/features.csv.zip')
stores_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/stores.csv')

train_df['Date'] = pd.to_datetime(train_df['Date'])
features_df['Date'] = pd.to_datetime(features_df['Date'])

weekly_sales = (
    train_df.groupby('Date', as_index=False)['Weekly_Sales']
    .sum()
    .sort_values('Date')
    .reset_index(drop=True)
)

holiday_flags = features_df[['Date', 'IsHoliday']].drop_duplicates(subset='Date')
weekly_sales = weekly_sales.merge(holiday_flags, on='Date', how='left')
weekly_sales['IsHoliday'] = weekly_sales['IsHoliday'].fillna(False)

weekly_sales = weekly_sales.set_index('Date')
weekly_sales.index.freq = 'W-FRI'  # Walmart მონაცემები კვირეულია, პარასკევებზე

print(weekly_sales.shape)
weekly_sales.head()

(143, 2)


,Weekly_Sales,IsHoliday
Date,,
2010-02-05,49750740.50,False
2010-02-12,48336677.63,True
2010-02-19,48276993.78,False
2010-02-26,43968571.13,False
2010-03-05,46871470.30,False


In [7]:
VAL_WEEKS = 12

train_series = weekly_sales.iloc[:-VAL_WEEKS]
val_series = weekly_sales.iloc[-VAL_WEEKS:]

print(f"Train: {train_series.index.min()} -> {train_series.index.max()} ({len(train_series)} კვირა)")
print(f"Val:   {val_series.index.min()} -> {val_series.index.max()} ({len(val_series)} კვირა)")

Train: 2010-02-05 00:00:00 -> 2012-08-03 00:00:00 (131 კვირა)
Val:   2012-08-10 00:00:00 -> 2012-10-26 00:00:00 (12 კვირა)


In [8]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [9]:
import mlflow
import mlflow.statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mlflow.set_experiment("Walmart-Sales-Forecasting")

runs_config = [
    {"name": "arima_1_1_1",        "order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    {"name": "arima_2_1_2",        "order": (2, 1, 2), "seasonal_order": (0, 0, 0, 0)},
    {"name": "sarima_1_1_1_52",    "order": (1, 1, 1), "seasonal_order": (1, 1, 1, 52)},
    {"name": "sarima_2_1_1_52",    "order": (2, 1, 1), "seasonal_order": (1, 0, 1, 52)},
]

results = []

for cfg in runs_config:
    with mlflow.start_run(run_name=cfg["name"]):
        mlflow.log_params({
            "model_type": "SARIMA" if any(cfg["seasonal_order"][:3]) else "ARIMA",
            "order": cfg["order"],
            "seasonal_order": cfg["seasonal_order"],
            "val_weeks": VAL_WEEKS,
        })

        model = SARIMAX(
            train_series['Weekly_Sales'],
            order=cfg["order"],
            seasonal_order=cfg["seasonal_order"],
            enforce_stationarity=False,
            enforce_invertibility=False,
        )

        fit_result = model.fit(disp=False, low_memory=True)


        train_pred = fit_result.fittedvalues

        train_wmae = wmae(
            train_series['Weekly_Sales'].values[52:],
            train_pred.values[52:],
            train_series['IsHoliday'].values[52:],
        )

        forecast = fit_result.get_forecast(steps=VAL_WEEKS)
        val_pred = forecast.predicted_mean.values

        val_wmae = wmae(
            val_series['Weekly_Sales'].values,
            val_pred,
            val_series['IsHoliday'].values,
        )

        mlflow.log_metrics({
            "train_wmae": train_wmae,
            "val_wmae": val_wmae,
            "aic": fit_result.aic,
            "bic": fit_result.bic,
        })

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(train_series.index[-30:], train_series['Weekly_Sales'].values[-30:], label="train (last 30w)")
        ax.plot(val_series.index, val_series['Weekly_Sales'].values, label="val actual")
        ax.plot(val_series.index, val_pred, label="val forecast", linestyle="--")
        ax.set_title(cfg["name"])
        ax.legend()

        fig_path = f"/content/{cfg['name']}_forecast.png"
        fig.savefig(fig_path)
        mlflow.log_artifact(fig_path)
        plt.close(fig)

        mlflow.statsmodels.log_model(fit_result, artifact_path="model")

        results.append({**cfg, "train_wmae": train_wmae, "val_wmae": val_wmae})

        avg_val_sales = val_series['Weekly_Sales'].mean()
        val_pct_error = (val_wmae / avg_val_sales) * 100
        print(f"{cfg['name']}: train_wmae={train_wmae:,.2f}, val_wmae={val_wmae:,.2f} (~{val_pct_error:.2f}% error)")

results_df = pd.DataFrame(results)
results_df

2026/07/12 18:45:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


arima_1_1_1: train_wmae=3,885,832.07, val_wmae=1,334,767.75 (~2.89% error)
🏃 View run arima_1_1_1 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9/runs/31a6778d50f846b8a2df83a867c3e826
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
2026/07/12 18:45:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


arima_2_1_2: train_wmae=3,284,790.51, val_wmae=1,442,853.89 (~3.12% error)
🏃 View run arima_2_1_2 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9/runs/9b486dab4df54547958bd49c4e9fb8d0
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
2026/07/12 18:45:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sarima_1_1_1_52: train_wmae=6,065,427.54, val_wmae=678,262.86 (~1.47% error)
🏃 View run sarima_1_1_1_52 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9/runs/85fe869afba54db284d399ced3bfc1f0
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9


2026/07/12 18:46:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sarima_2_1_1_52: train_wmae=1,533,939.54, val_wmae=1,374,021.74 (~2.97% error)
🏃 View run sarima_2_1_1_52 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9/runs/ee5ca421ba12469cb04a6c63a371e07c
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/9


,name,order,seasonal_order,train_wmae,val_wmae
0,arima_1_1_1,"(1, 1, 1)","(0, 0, 0, 0)",3.885832e+06,1.334768e+06
1,arima_2_1_2,"(2, 1, 2)","(0, 0, 0, 0)",3.284791e+06,1.442854e+06
2,sarima_1_1_1_52,"(1, 1, 1)","(1, 1, 1, 52)",6.065428e+06,6.782629e+05
3,sarima_2_1_1_52,"(2, 1, 1)","(1, 0, 1, 52)",1.533940e+06,1.374022e+06


In [10]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip')
test_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/test.csv.zip')

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

train_df['Week'] = train_df['Date'].dt.isocalendar().week
test_df['Week'] = test_df['Date'].dt.isocalendar().week

print(f"Test set shape: {test_df.shape}")


historical_means = train_df.groupby(['Store', 'Dept', 'Week', 'IsHoliday'])['Weekly_Sales'].mean().reset_index()

submission_df = pd.merge(test_df, historical_means, on=['Store', 'Dept', 'Week', 'IsHoliday'], how='left')

backup_means = train_df.groupby(['Store', 'Dept'])['Weekly_Sales'].mean().reset_index()
backup_means.rename(columns={'Weekly_Sales': 'Backup_Sales'}, inplace=True)

submission_df = pd.merge(submission_df, backup_means, on=['Store', 'Dept'], how='left')
submission_df['Weekly_Sales'] = submission_df['Weekly_Sales'].fillna(submission_df['Backup_Sales'])

global_mean = train_df['Weekly_Sales'].mean()
submission_df['Weekly_Sales'] = submission_df['Weekly_Sales'].fillna(global_mean)

submission_df['Id'] = (
    submission_df['Store'].astype(str) + '_' +
    submission_df['Dept'].astype(str) + '_' +
    submission_df['Date'].dt.strftime('%Y-%m-%d')
)

final_submission = submission_df[['Id', 'Weekly_Sales']]

final_submission.to_csv('/submission.csv', index=False)

print("Submission file successfully created at /submission.csv!")
print(final_submission.head())

Test set shape: (115064, 5)
Submission file successfully created at /submission.csv!
               Id  Weekly_Sales
0  1_1_2012-11-02     37062.470
1  1_1_2012-11-09     19119.465
2  1_1_2012-11-16     19301.750
3  1_1_2012-11-23     19865.770
4  1_1_2012-11-30     23905.525
